# Bronze Layer Ingestion
Ingest raw JSON files from source system into Delta Lake (Bronze Tables).

In [1]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import input_file_name, current_timestamp
from delta import configure_spark_with_delta_pip
from dotenv import load_dotenv

# Load Environment
load_dotenv("../../.env")
bronze_source = os.getenv("BRONZE_SOURCE_PATH")
print(f"Source Path: {bronze_source}")

# Fix JAVA_HOME spaces on Windows
if sys.platform.startswith('win'):
    java_home = os.environ.get('JAVA_HOME', '')
    if ' ' in java_home and os.path.exists(java_home):
        try:
            import win32api
            os.environ['JAVA_HOME'] = win32api.GetShortPathName(java_home)
        except ImportError:
            pass
    if not os.environ.get('HADOOP_HOME'):
        os.environ['HADOOP_HOME'] = "C:\\hadoop"

# Initialize Spark (Config matches UserSilver.ipynb)
builder = SparkSession.builder \
    .appName("Yelp_Bronze_Ingest") \
    .config("spark.driver.memory", "4g") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.databricks.delta.schema.autoMerge.enabled", "true") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print(f"Spark Version: {spark.version}")

Source Path: D:\Project\delta_lake\bronze
Spark Version: 3.5.1


In [2]:
def ingest_table(source_dir, file_name, delta_dir, table_name):
    src_path = os.path.join(source_dir, file_name)
    dst_path = os.path.join(delta_dir, table_name)
    
    if not os.path.exists(src_path):
        print(f"[SKIP] Source not found: {src_path}")
        return
        
    print(f"[{table_name}] Reading {src_path}...")
    df = spark.read.json(src_path)
    
    df = df.withColumn("_ingest_timestamp", current_timestamp()) \
           .withColumn("_source_file", input_file_name())
           
    print(f"[{table_name}] Writing to {dst_path}...")
    df.write.format("delta").mode("overwrite").save(dst_path)
    print(f"[{table_name}] Success. Rows: {df.count()}")


In [3]:
# Execute Ingestion
# Define Output path relative to project root
root_dir = os.path.abspath("../../")
delta_root = os.path.join(root_dir, "data", "bronze_delta")

# 1. Business
ingest_table(bronze_source, "yelp_academic_dataset_business.json", delta_root, "business")

# 2. User
ingest_table(bronze_source, "yelp_academic_dataset_user.json", delta_root, "user")

# 3. Checkin (Small enough to run)
ingest_table(bronze_source, "yelp_academic_dataset_checkin.json", delta_root, "checkin")

[business] Reading D:\Project\delta_lake\bronze\yelp_academic_dataset_business.json...
[business] Writing to d:\Project\YELP_RS\data\bronze_delta\business...
[business] Success. Rows: 150346
[user] Reading D:\Project\delta_lake\bronze\yelp_academic_dataset_user.json...
[user] Writing to d:\Project\YELP_RS\data\bronze_delta\user...
[user] Success. Rows: 1987897
[checkin] Reading D:\Project\delta_lake\bronze\yelp_academic_dataset_checkin.json...
[checkin] Writing to d:\Project\YELP_RS\data\bronze_delta\checkin...
[checkin] Success. Rows: 131930
